# Bar Coverage & Alignment Validation Report

This notebook visualizes the outputs generated by:

- `src/data_pipeline/analysis/validate_bars.py`

It loads the latest CSV reports from `data/validation/` and produces presentation-friendly tables and plots.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from data_pipeline.core.config import load_config

cfg = load_config(None)
base_out = Path(cfg.output_dir)
validation_dir = base_out / "validation"

validation_dir

In [ ]:
# Load the latest run reports written by validate_bars.py
def latest_csv(prefix: str) -> Path:
    files = sorted(validation_dir.glob(f"{prefix}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f"No files found for prefix={prefix!r} under {validation_dir}")
    return files[0]

coverage_path = latest_csv("bar_coverage_report")
alignment_path = latest_csv("bar_alignment_report")

coverage_path, alignment_path

In [ ]:
coverage = pd.read_csv(coverage_path)
alignment = pd.read_csv(alignment_path)

coverage.head(), alignment.head()

## 1) Bar coverage summary

Coverage is computed by `validate_bars.py` as:

- expected buckets based on `(end_ms - start_ms) // period_ms + 1`
- observed buckets = number of unique `timestamp_ms`
- missing buckets = expected - observed
- coverage % = observed / expected * 100


In [ ]:
# Present a compact table
coverage_summary = (
    coverage.groupby(["source", "timeframe"], as_index=False)
    .agg(mean_coverage_pct=("coverage_pct", "mean"),
         min_coverage_pct=("coverage_pct", "min"),
         worst_symbol=("coverage_pct", lambda s: coverage.loc[s.idxmin(), "symbol_safe"]))
    .sort_values(["source", "timeframe"])
)

coverage_summary

In [ ]:
# Coverage plot: coverage_pct by timeframe, split by source
plt.figure(figsize=(10, 5))
for src, df in coverage.groupby("source"):
    df_plot = df.copy()
    df_plot = df_plot.sort_values("timeframe")
    plt.plot(df_plot["timeframe"], df_plot["coverage_pct"], marker="o", label=src)

plt.title("Bar Coverage % vs Timeframe")
plt.ylabel("coverage_pct")
plt.xlabel("timeframe")
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

# Worst gaps
coverage.sort_values("max_gap_bars", ascending=False).head(10)[["symbol_safe","timeframe","source","coverage_pct","max_gap_bars","missing_bars"]]

## 2) Alignment validation (trades vs exchange)

For matching `(symbol_safe, timeframe)` buckets, `validate_bars.py` computes summary deltas:

- `mean_abs_open_diff`
- `p95_abs_open_diff`
- `mean_abs_volume_diff`
- `max_abs_close_diff`
- `% bars within tolerance` for open and volume


In [ ]:
alignment

In [ ]:
# Top alignment issues
alignment.sort_values("p95_abs_open_diff", ascending=False).head(10)

In [ ]:
# Plot: match rate vs timeframe
plt.figure(figsize=(10, 5))
for tf, df in alignment.groupby("timeframe"):
    plt.scatter([tf]*len(df), df["match_rate_pct"], alpha=0.9)

plt.title("Alignment match rate (overlap coverage) by timeframe")
plt.ylabel("match_rate_pct")
plt.xlabel("timeframe")
plt.tight_layout()
plt.show()

alignment[["symbol_safe","timeframe","matched_bars","overlap_expected_bars","match_rate_pct","open_within_tol_pct","volume_within_tol_pct"]].sort_values(["timeframe","symbol_safe"])

## Notes (methodology alignment)

- The notebook intentionally uses the same filenames and column names as `validate_bars.py`.
- The main goal is reproducible, evidence-ready diagnostics that you can screenshot or export.
